# SWAN Model Runner: Setup, Execution & Output Parsing

---

## Overview

The **SWAN** (Simulating WAves Nearshore) model is a third-generation spectral wave model developed at Delft University of Technology. It solves the spectral action balance equation to compute wave parameters in coastal waters, accounting for:

- Wind generation
- Whitecapping dissipation
- Bottom friction
- Depth-induced breaking
- Refraction and shoaling
- Nonlinear wave-wave interactions

This notebook provides a **Python interface** for:

1. **Modifying SWAN input files** to inject custom output commands
2. **Invoking the SWAN executable** via `subprocess`
3. **Parsing SWAN output files** (TABLE text and BLOCK `.MAT` binary formats)

These utilities are used by the coupled site-selection and optimization notebooks later in the workflow.

### Prerequisites

- SWAN must be compiled and available on your system PATH (see the repository README for installation instructions)
- Dependencies: `numpy`, `scipy` (for `.MAT` files)

### SWAN Input File Structure

A SWAN input file (`INPUT`) is a sequence of keyword commands:

```
PROJECT 'name' ...
CGRID ...          ! Computational grid
INPgrid BOTtom ... ! Bathymetry input specification
READinp BOTtom ... ! Read bathymetry file
BOUnd ...          ! Boundary conditions
GEN3 ...           ! Physics: wave generation
...
COMPUTE ...        ! Run the simulation
STOP
```

Our runner inserts output commands (TABLE, BLOCK, POINTS) **just before the COMPUTE line**.

> **Where the code lives.** All functions shown below are imported from `code/swan_runner.py`, the same module every other notebook uses. The cells display its source with `show_source(...)` instead of re-defining the functions, so the notebook and the library can never drift apart. Edit `code/swan_runner.py` to change behaviour.


In [ ]:
# --- bootstrap: locate the repository and make code/ importable ---
import os, sys
REPO_ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(REPO_ROOT, 'code')):
    if os.path.dirname(REPO_ROOT) == REPO_ROOT:
        raise FileNotFoundError('Repository root not found (no code/ directory). '
                                'Start Jupyter from inside the repository.')
    REPO_ROOT = os.path.dirname(REPO_ROOT)
CODE_DIR = os.path.join(REPO_ROOT, 'code')
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
# --- end bootstrap ---

import inspect
import swan_runner
from swan_runner import (prepare_input, write_points_file, run_swan,
                         read_table, read_mat_block, extract_block_variable,
                         list_block_variables, check_swan_available)


def show_source(*objs):
    """Display the source of functions from code/swan_runner.py."""
    text = '\n\n'.join(inspect.getsource(o) for o in objs)
    try:
        from IPython.display import Markdown, display
        display(Markdown(f"~~~python\n{text}~~~"))
    except ImportError:
        print(text)

print(f"Using {swan_runner.__file__}")


## 1. Prepare SWAN Input File

The `prepare_input` function takes an existing SWAN input template, copies it, and inserts additional commands (e.g., output requests) immediately before the `COMPUTE` line. This allows us to programmatically request specific outputs without manually editing the input file each time.

**How it works:**
1. Read all lines from the template
2. Find the line starting with `COMP` (the COMPUTE command)
3. Insert the new commands just before it
4. Write the modified file

In [ ]:
show_source(prepare_input)


## 2. Write Points File

SWAN can output results at specific geographic locations defined in a **points file**. Each line contains a longitude-latitude pair. This is used by the AHP-TOPSIS site-selection workflow to extract wave parameters at candidate deployment sites.

In [ ]:
show_source(write_points_file)


## 3. Execute SWAN

The `run_swan` function invokes the SWAN executable as a subprocess. Key parameters:

| Parameter | Description |
|-----------|-------------|
| `model_dir` | Working directory containing all SWAN files (INPUT, bathymetry, boundary files, etc.) |
| `input_name` | Name of the input file (without extension) |
| `swan_cmd` | The SWAN executable command (`swanrun`, `swan.exe`, etc.) |
| `nproc` | Number of OpenMP threads for parallel execution |
| `timeout` | Maximum runtime in seconds before the process is killed |

After execution, SWAN writes a `norm_end` file indicating normal completion. If this file is missing and the return code is non-zero, the function raises an error.

In [ ]:
show_source(run_swan)


## 4. Output Readers

SWAN produces two main output formats:

### 4.1 TABLE Output

A text file with `%`-prefixed header lines followed by space-delimited numerical data. Each row corresponds to a point at a timestep. The variable names (e.g., `Hsig`, `RTpeak`, `Dir`, `Depth`) are extracted from the header.

In [ ]:
show_source(read_table)


### 4.2 BLOCK Output (`.MAT` format)

SWAN can write 2-D gridded output in MATLAB `.MAT` format (via `LAY-OUT 3`). Each variable at each timestep is stored as a separate matrix with a name like `Hsig_20240101_030000`. We use `scipy.io.loadmat` to read these files and a helper function to stack all timesteps into a 3-D array `(time, ny, nx)`.

In [ ]:
show_source(read_mat_block, extract_block_variable, list_block_variables)


## 5. Utility: Check SWAN Availability

Before attempting to run SWAN, we can check whether the executable is found on the system PATH.

In [ ]:
# Quick check
if check_swan_available():
    print("SWAN is available on your system.")
else:
    print("WARNING: 'swanrun' not found on PATH.")
    print("The coupled notebooks (site selection, GA optimization) require SWAN.")
    print("See the repository README for installation instructions.")


## 6. Example: Inspect a SWAN Input File

Let's read the California Bay SWAN input file to see its structure. This helps understand what grid, physics, and boundary settings are configured.

In [ ]:
# Demonstrate the utilities by reading the SWAN input for a selected region.

from regions import get_region, list_regions, summarize  # noqa: E402

REGION = os.environ.get('SWAN_REGION', 'california_bay')
cfg = get_region(REGION)
print(summarize(REGION))

input_path = cfg.swan_input_path
if os.path.exists(input_path):
    with open(input_path) as f:
        content = f.read()
    print("=" * 65)
    print(f"SWAN input file for {cfg.display_name}: {os.path.basename(input_path)}")
    print("=" * 65)
    print(content)
else:
    print(f"Input file not found at: {input_path}")

# Per-region figures directory for this notebook
FIGS_DIR = cfg.figs_dir('02_model_setup')
print(f'Figures will be saved to: {FIGS_DIR}')


## Summary

This notebook walked through the core SWAN interface functions in `code/swan_runner.py`:

| Function | Purpose |
|----------|--------|
| `prepare_input()` | Inject output commands into a SWAN input template |
| `write_points_file()` | Create a SWAN-format points file for output locations |
| `run_swan()` | Execute the SWAN model as a subprocess |
| `read_table()` | Parse SWAN TABLE text output |
| `read_mat_block()` | Read SWAN BLOCK `.MAT` binary output |
| `extract_block_variable()` | Stack timestep matrices into a 3-D array |
| `check_swan_available()` | Verify SWAN is installed |

These functions are imported by the downstream notebooks:
- `03_site_selection/swan_coupled_ahp_topsis.ipynb`
- `04_optimization/swan_coupled_ga_array.ipynb`